<a href="https://colab.research.google.com/github/KatreenGhobrial/RepoCloudComputing/blob/main/Tirgulim/tirgul7/Tutorial_7_microservices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("KPI needed:")
print("1. Performance -––> response time")
print("by calculating the start time and the end time of the qury request ")
print("Data synchronization from the IoT sensors to the application will be performed with a latency of no more than 5 seconds.\n")
print("2. Availability & Reliability")
print("The remote irrigation system will operate with 99.9% uptime to ensure that plants are not harmed.\n")
print("3. User experience")
print("pop up window that rate the experience of the user")
print("there will be a 'contact us' section on the website where users can click and get in touch with us\n")

KPI needed:
1. Performance -––> response time
by calculating the start time and the end time of the qury request 
Data synchronization from the IoT sensors to the application will be performed with a latency of no more than 5 seconds.

2. Availability & Reliability
The remote irrigation system will operate with 99.9% uptime to ensure that plants are not harmed.

3. User experience
pop up window that rate the experience of the user
there will be a 'contact us' section on the website where users can click and get in touch with us



fatal: not in a git directory


In [ ]:
import requests

FIREBASE_DB_URL = "https://plant-70648-default-rtdb.firebaseio.com"

class SearchService:
    def __init__(self):
        pass

    def get_document_info(self, doc_id):
        resp = requests.get(f"{FIREBASE_DB_URL}/documents/{doc_id}.json")
        if resp.status_code == 200 and resp.json():
            return resp.json()
        return {"url": "Unknown", "title": "Unknown"}

    def search(self, query):
        print(f"\n[Searcher] Executing search for: '{query}'")

        operator = "AND" if " AND " in query else "OR"

        clean_query = query.replace(" AND ", " ").replace(" OR ", " ").lower()
        search_terms = clean_query.split()

        doc_scores = {}
        term_docs_list = []

        for term in search_terms:
            resp = requests.get(f"{FIREBASE_DB_URL}/inverted_index/{term}.json")
            if resp.status_code == 200 and resp.json():
                documents_with_term = resp.json()

                term_docs_list.append(set(documents_with_term.keys()))

                for doc_id, count in documents_with_term.items():
                    if doc_id not in doc_scores:
                        doc_scores[doc_id] = 0
                    doc_scores[doc_id] += count
            else:
                term_docs_list.append(set())

        if not doc_scores or not term_docs_list:
            print("[Searcher] No results found.")
            return []

        valid_doc_ids = set(doc_scores.keys())

        if operator == "AND":
            valid_doc_ids = set.intersection(*term_docs_list)

        if not valid_doc_ids:
            print(f"[Searcher] No documents matched the {operator} condition.")
            return []

        results = []
        for doc_id in valid_doc_ids:
            score = doc_scores[doc_id]
            doc_info = self.get_document_info(doc_id)
            results.append({
                "score": score,
                "title": doc_info.get("title"),
                "url": doc_info.get("url")
            })

        sorted_results = sorted(results, key=lambda item: item["score"], reverse=True)
        return sorted_results

def main():
    searcher = SearchService()


    print("\n--- Test 1: OR Operator (Implicit) ---")
    results_or = searcher.search("plant virus")
    for index, res in enumerate(results_or, 1):
        print(f"{index}. Score: {res['score']} | {res['title']}")

    print("\n--- Test 2: AND Operator (Explicit) ---")
    results_and = searcher.search("plant AND virus")
    for index, res in enumerate(results_and, 1):
        print(f"{index}. Score: {res['score']} | {res['title']}")

if __name__ == "__main__":
    main()


--- Test 1: OR Operator (Implicit) ---

[Searcher] Executing search for: 'plant virus'
1. Score: 93 | Wikipedia - Plant Pathology
2. Score: 18 | GardenTech - Fungal Disease

--- Test 2: AND Operator (Explicit) ---

[Searcher] Executing search for: 'plant AND virus'
1. Score: 93 | Wikipedia - Plant Pathology


In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import hashlib

FIREBASE_DB_URL = "https://plant-70648-default-rtdb.firebaseio.com"

class IndexerService:
    def __init__(self):
        self.stop_words = {'a', 'an', 'the', 'and', 'or', 'in', 'on', 'at', 'to', 'of', 'for', 'is', 'are'}

    def generate_doc_id(self, url):
        return hashlib.md5(url.encode()).hexdigest()

    def fetch_page(self, url):
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            return BeautifulSoup(response.text, "html.parser")
        return None

    def index_url(self, url, title):
        print(f"[Indexer] Starting to index: {url}")
        soup = self.fetch_page(url)
        if not soup:
            print(f"[Indexer] Failed to load {url}")
            return False

        text = soup.get_text()
        words = re.findall(r'\b[a-zA-Z]+\b', text.lower())

        word_counts = {}
        for word in words:
            if word not in self.stop_words and len(word) > 2:
                word_counts[word] = word_counts.get(word, 0) + 1

        doc_id = self.generate_doc_id(url)

        doc_info = {"url": url, "title": title}
        requests.put(f"{FIREBASE_DB_URL}/documents/{doc_id}.json", json=doc_info)

        for word, count in word_counts.items():
            firebase_url = f"{FIREBASE_DB_URL}/inverted_index/{word}.json"
            requests.patch(firebase_url, json={doc_id: count})

        print(f"[Indexer] Successfully indexed {len(word_counts)} unique words from {title}")
        return True


class SearchService:
    def __init__(self):
        pass

    def get_document_info(self, doc_id):
        resp = requests.get(f"{FIREBASE_DB_URL}/documents/{doc_id}.json")
        if resp.status_code == 200 and resp.json():
            return resp.json()
        return {"url": "Unknown", "title": "Unknown"}

    def search(self, query):
        print(f"\n[Searcher] Executing search for: '{query}'")
        search_terms = query.lower().split()

        doc_scores = {}

        for term in search_terms:
            resp = requests.get(f"{FIREBASE_DB_URL}/inverted_index/{term}.json")
            if resp.status_code == 200 and resp.json():
                documents_with_term = resp.json()

                for doc_id, count in documents_with_term.items():
                    if doc_id not in doc_scores:
                        doc_scores[doc_id] = 0
                    doc_scores[doc_id] += count

        if not doc_scores:
            print("[Searcher] No results found.")
            return []

        sorted_docs = sorted(doc_scores.items(), key=lambda item: item[1], reverse=True)

        results = []
        for doc_id, score in sorted_docs:
            doc_info = self.get_document_info(doc_id)
            results.append({
                "score": score,
                "title": doc_info.get("title"),
                "url": doc_info.get("url")
            })

        return results


def main():
    indexer = IndexerService()
    searcher = SearchService()

    print("--- Phase 1: Ingestion / Indexing ---")
    indexer.index_url(
        url="https://www.gardentech.com/blog/pest-id-and-prevention/keep-your-garden-free-from-fungal-disease",
        title="GardenTech - Fungal Disease"
    )
    indexer.index_url(
        url="https://en.wikipedia.org/wiki/Plant_pathology",
        title="Wikipedia - Plant Pathology"
    )

    print("\n--- Phase 2: Searching & Ranking ---")
    query = "plant disease fungi"
    results = searcher.search(query)

    print("\nSearch Results (Ranked by Word Frequency Score):")
    for index, res in enumerate(results, 1):
        print(f"{index}. Score: {res['score']} | {res['title']}")
        print(f"   Link: {res['url']}\n")

if __name__ == "__main__":
    main()

--- Phase 1: Ingestion / Indexing ---
[Indexer] Starting to index: https://www.gardentech.com/blog/pest-id-and-prevention/keep-your-garden-free-from-fungal-disease
[Indexer] Successfully indexed 488 unique words from GardenTech - Fungal Disease
[Indexer] Starting to index: https://en.wikipedia.org/wiki/Plant_pathology
[Indexer] Failed to load https://en.wikipedia.org/wiki/Plant_pathology

--- Phase 2: Searching & Ranking ---

[Searcher] Executing search for: 'plant disease fungi'

Search Results (Ranked by Word Frequency Score):
1. Score: 114 | Wikipedia - Plant Pathology
   Link: https://en.wikipedia.org/wiki/Plant_pathology

2. Score: 40 | GardenTech - Fungal Disease
   Link: https://www.gardentech.com/blog/pest-id-and-prevention/keep-your-garden-free-from-fungal-disease



In [ ]:
# user_service.py
class UserService:
    def __init__(self):
        self.users = {
            '1': {'id': '1', 'name': 'John Doe', 'email': 'john@example.com'},
            '2': {'id': '2', 'name': 'Jane Doe', 'email': 'jane@example.com'}
        }

    def get_user(self, user_id):
        return self.users.get(user_id, {})

# index_service.py
class IndexService:
    def __init__(self):
        self.documents = {}
        self.index = {}

    def add_document(self, doc_data):
        """Add a document to the index"""
        doc_id = str(len(self.documents) + 1)
        self.documents[doc_id] = {**doc_data, 'id': doc_id}

        # Create inverted index
        words = doc_data['content'].lower().split()
        for word in words:
            if word not in self.index:
                self.index[word] = set()
            self.index[word].add(doc_id)

        return self.documents[doc_id]

    def get_document(self, doc_id):
        """Retrieve a document by ID"""
        return self.documents.get(doc_id)

    def search_word(self, word):
        """Find documents containing a word"""
        word = word.lower()
        return list(self.index.get(word, set()))

# query_service.py
class QueryService:
    def __init__(self, index_service):
        self.index_service = index_service
        self.queries = {}

    def create_query(self, query_data):
        """Create and execute a search query"""
        try:
            query_id = str(len(self.queries) + 1)
            search_terms = query_data['terms']

            # Find matching documents
            results = set()
            for term in search_terms:
                doc_ids = self.index_service.search_word(term)
                if not results:
                    results = set(doc_ids)
                else:
                    results &= set(doc_ids)

            # Create query record
            query = {
                'id': query_id,
                'terms': search_terms,
                'results': list(results),
                'timestamp': query_data.get('timestamp', 'now')
            }
            self.queries[query_id] = query
            return query

        except Exception as e:
            return {'error': str(e)}

# result_service.py
class ResultService:
    def __init__(self, index_service, query_service):
        self.index_service = index_service
        self.query_service = query_service
        self.results = {}

    def format_results(self, query_id):
        """Format search results for display"""
        try:
            query = self.query_service.queries.get(query_id)
            if not query:
                return {'error': 'Query not found'}

            formatted_results = []
            for doc_id in query['results']:
                doc = self.index_service.get_document(doc_id)
                if doc:
                    formatted_results.append({
                        'doc_id': doc_id,
                        'title': doc['title'],
                        'snippet': doc['content'][:100] + '...'
                    })

            result_id = str(len(self.results) + 1)
            result = {
                'id': result_id,
                'query_id': query_id,
                'formatted_results': formatted_results,
                'count': len(formatted_results)
            }
            self.results[result_id] = result
            return result

        except Exception as e:
            return {'error': str(e)}

# main.py
def main():
    # Initialize services
    index_service = IndexService()
    query_service = QueryService(index_service)
    result_service = ResultService(index_service, query_service)

    # Add sample documents
    doc1 = index_service.add_document({
        'title': 'Python Programming',
        'content': 'Python is a popular programming language for cloud computing'
    })
    doc2 = index_service.add_document({
        'title': 'Cloud Services',
        'content': 'Cloud computing enables scalable microservices architecture'
    })
    print(f"Added documents: {doc1['id']}, {doc2['id']}")

    # Create and execute a search query
    query = query_service.create_query({
        'terms': ['cloud', 'computing']
    })
    print(f"Query results: {query}")

    # Format the results
    formatted_results = result_service.format_results(query['id'])
    print(f"Formatted results: {formatted_results}")

if __name__ == "__main__":
    main()

# Example test cases
def test_search():
    # Initialize services
    index_service = IndexService()
    query_service = QueryService(index_service)
    result_service = ResultService(index_service, query_service)

    # Test document indexing
    doc = index_service.add_document({
        'title': 'Test Document',
        'content': 'This is a test document about microservices'
    })
    assert doc['id'] == '1'

    # Test search functionality
    query = query_service.create_query({
        'terms': ['test', 'microservices']
    })
    assert len(query['results']) == 1

    # Test result formatting
    results = result_service.format_results(query['id'])
    assert results['count'] == 1
    assert 'test document' in results['formatted_results'][0]['snippet'].lower()

if __name__ == "__main__":
    test_search()

Added documents: 1, 2
Query results: {'id': '1', 'terms': ['cloud', 'computing'], 'results': ['1', '2'], 'timestamp': 'now'}
Formatted results: {'id': '1', 'query_id': '1', 'formatted_results': [{'doc_id': '1', 'title': 'Python Programming', 'snippet': 'Python is a popular programming language for cloud computing...'}, {'doc_id': '2', 'title': 'Cloud Services', 'snippet': 'Cloud computing enables scalable microservices architecture...'}], 'count': 2}


In [ ]:


"""
Function as a Service (FaaS) Architecture Demo: Search Engine Implementation

This code demonstrates key FaaS principles through a simple search engine implementation.
It simulates how serverless functions would operate in a cloud environment.

Key FaaS Characteristics Demonstrated:
1. Stateless Functions - Each invocation is independent
2. Event-Driven Architecture - Functions respond to specific triggers
3. Single Responsibility - Each function performs one specific task
4. Automatic Scaling (simulated) - Functions can handle multiple concurrent requests
"""

class IndexerFunction:

    def __init__(self):
        # In real FaaS, this would be external storage
        self.index = {}

    def handle(self, event):
        """
        Function entry point - similar to AWS Lambda handler

        Args:
            event (dict): Contains document to be indexed
                {
                    'document_id': str - Unique document identifier
                    'content': str - Document content to index
                }

        Returns:
            dict: Indexing operation results
                {
                    'status': str - Operation status
                    'indexed_words': int - Number of words processed
                }
        """
        doc_id = event['document_id']
        content = event['content'].lower().split()

        # Build inverted index
        for word in content:
            if word not in self.index:
                self.index[word] = set()
            self.index[word].add(doc_id)

        return {
            'status': 'success',
            'indexed_words': len(content)
        }

class SearcherFunction:
    """
    Simulates a FaaS search function.

    FaaS Characteristics:
    - Event-driven: Responds to search requests
    - Stateless: Each search is independent
    - Scalable: Multiple instances can handle concurrent searches


    """
    def __init__(self, index_service):
        self.index = index_service.index

    def handle(self, event):
        """
        Search function entry point

        Args:
            event (dict): Contains search parameters
                {
                    'query': str - Search terms
                }

        Returns:
            dict: Search results
                {
                    'status': str - Operation status
                    'results': list - Matching document IDs
                }

        Note: In real FaaS:
        - Would include error handling
        - Would implement timeouts
        - Would include logging/monitoring
        """
        terms = event['query'].lower().split()
        results = set()

        for term in terms:
            if term in self.index:
                if not results:
                    results = self.index[term].copy()
                else:
                    results &= self.index[term]

        return {
            'status': 'success',
            'results': list(results)
        }

class FaaSSimulator:
    """
    Simulates a FaaS environment.

    Demonstrates:
    - Function isolation
    - Event-based invocation
    - Resource management (simulated)
    - Function routing

    In real FaaS platforms (like AWS Lambda):
    - Functions run in isolated containers
    - Resources are automatically managed
    - Scaling happens automatically
    - Includes monitoring and logging
    """
    def __init__(self):
        self.indexer = IndexerFunction()
        self.searcher = SearcherFunction(self.indexer)
        self.invocations = 0

    def invoke(self, function_name, event):
        """
        Simulates FaaS function invocation

        Args:
            function_name (str): Name of function to invoke
            event (dict): Event data for the function

        Real FaaS differences:
        - Would spawn new container/instance
        - Would handle concurrent requests
        - Would implement timeout limits
        - Would include error handling
        """
        self.invocations += 1

        if function_name == 'indexer':
            return self.indexer.handle(event)
        elif function_name == 'searcher':
            return self.searcher.handle(event)
        else:
            raise ValueError(f"Unknown function: {function_name}")

def demonstrate_faas():
    """
    Demonstrates key FaaS concepts through example usage

    Shows:
    1. Event-driven invocation
    2. Function independence
    3. Scalability potential
    """
    # Test data setup
    test_documents = [
        {
            'document_id': 'doc1',
            'content': 'Python is a popular programming language for cloud computing'
        },
        {
            'document_id': 'doc2',
            'content': 'Cloud computing enables scalable microservices architecture'
        }
    ]

    # Initialize FaaS environment
    faas = FaaSSimulator()

    # Demonstrate event-driven invocation
    print("1. Event-Driven Invocation:")
    for doc in test_documents:
        result = faas.invoke('indexer', doc)
        print(f"  Indexed document {doc['document_id']}: {result}")

    # Demonstrate independent function calls
    print("\n2. Independent Function Calls:")
    search_queries = [
        {'query': 'cloud computing'},
        {'query': 'python programming'}
    ]

    for query in search_queries:
        result = faas.invoke('searcher', query)
        print(f"  Search results for '{query['query']}': {result}")

    # Demonstrate scalability concept
    print(f"\n3. Scalability Demonstration:")
    print(f"  Total function invocations: {faas.invocations}")
    print("  In real FaaS: These would execute in parallel with automatic scaling")

if __name__ == "__main__":
    demonstrate_faas()


1. Event-Driven Invocation:
  Indexed document doc1: {'status': 'success', 'indexed_words': 9}
  Indexed document doc2: {'status': 'success', 'indexed_words': 6}

2. Independent Function Calls:
  Search results for 'cloud computing': {'status': 'success', 'results': ['doc1', 'doc2']}
  Search results for 'python programming': {'status': 'success', 'results': ['doc1']}

3. Scalability Demonstration:
  Total function invocations: 4
  In real FaaS: These would execute in parallel with automatic scaling
